# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# 1. Clean missing values completely
spaceship = spaceship.dropna()





In [4]:
# 2. Extract the Deck letter from Cabin
spaceship['Cabin'] = spaceship['Cabin'].astype(str).str[0]



In [5]:
# 3. Drop non-predictive text IDs
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])



In [6]:
# 4. Convert all non-numerical features to dummy flags
spaceship = pd.get_dummies(spaceship, drop_first=True)

**Perform Train Test Split**

In [9]:
# Separate features and target label
X = spaceship.drop(columns=['Transported'])
y = spaceship['Transported']

# 80/20 data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [10]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# Initialize parallel bootstrap row bagging
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=15),
    n_estimators=100,
    bootstrap=True,
    random_state=42
)
bagging_clf.fit(X_train, y_train)

print("Bagging Train Score:", bagging_clf.score(X_train, y_train))
print("Bagging Test Score:", bagging_clf.score(X_test, y_test))


# Initialize parallel bootstrap row bagging
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=15),
    n_estimators=100,
    bootstrap=True,
    random_state=42
)
bagging_clf.fit(X_train, y_train)

print("Bagging Train Score:", bagging_clf.score(X_train, y_train))
print("Bagging Test Score:", bagging_clf.score(X_test, y_test))


Bagging Train Score: 0.9199470098410295
Bagging Test Score: 0.8033282904689864
Bagging Train Score: 0.9199470098410295
Bagging Test Score: 0.8033282904689864


- Random Forests

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest with feature subspace sampling
forest_clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
forest_clf.fit(X_train, y_train)

print("Random Forest Train Score:", forest_clf.score(X_train, y_train))
print("Random Forest Test Score:", forest_clf.score(X_test, y_test))


Random Forest Train Score: 0.924110522331567
Random Forest Test Score: 0.8018154311649016


- Gradient Boosting

In [12]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize sequential gradient residual error boosting
gb_clf = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
gb_clf.fit(X_train, y_train)

print("Gradient Boosting Train Score:", gb_clf.score(X_train, y_train))
print("Gradient Boosting Test Score:", gb_clf.score(X_test, y_test))


Gradient Boosting Train Score: 0.8565480696442089
Gradient Boosting Test Score: 0.8093797276853253


- Adaptive Boosting

In [13]:
from sklearn.ensemble import AdaBoostClassifier

# Initialize sequential sample-weight boosting 
ada_clf = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_clf.fit(X_train, y_train)

print("AdaBoost Train Score:", ada_clf.score(X_train, y_train))
print("AdaBoost Test Score:", ada_clf.score(X_test, y_test))


AdaBoost Train Score: 0.7842543527630583
AdaBoost Test Score: 0.7859304084720121


Which model is the best and why?

In [14]:
# Final Lab Conclusion:
# 1. Random Forest / Gradient Boosting perform best on this dataset (~79%-81% accuracy).
# 2. Why: Random Forest limits column dominance by picking random feature subsets at every node split, which cancels out individual tree overfitting.
# 3. Why: Gradient Boosting successfully reduces bias by forcing sequential trees to learn from the exact residual prediction errors of previous trees.
# 4. Scaling note: Unlike KNN, tree-based ensemble methods are scale-invariant, allowing spending metrics (like Spa, RoomService) to keep their raw variance power.


### Final Lab Analysis: Which model is the best and why?

1. **Top Performers:** Both **Random Forest** and **Gradient Boosting** consistently deliver the highest generalization accuracy on this dataset, hovering around the **~79% - 81% test accuracy** range.

2. **Why Random Forest Excelled:** It breaks variance and column dominance limits by forcing an extraction of a random subspace feature subset at every node split. This prevents a single hyper-correlated feature (like `Spa` or `CryoSleep`) from ruining structural choice logic across all individual baseline tree estimators.

3. **Why Gradient Boosting Excelled:** It focuses entirely on structural bias reduction. Rather than independent aggregation (like Bagging), it explicitly maps out new sequential stumps whose single job is to optimize against the raw residual error vectors left behind by prior operational passes.

4. **Data Scale-Invariance Advantage:** Tree-based ensemble structures are scale-invariant. Unlike distance algorithms (such as KNN), they do not demand normalization across varying scale features, allowing high-variance continuous spending features (e.g., `RoomService`, `Spa`) to maintain their authentic signal strength without getting compressed.
